# 第 23 课｜三种机器做同一个实验

只有当被比较的 experiment 真的是同一个 experiment 时，CPU、GPU 与 FPGA 的结果才有意义。

今天只问一个问题：

> **CPU/GPU/FPGA benchmark 在什么东西保持不变时才算公平？**

本课主要新概念：**benchmark comparability——同 model、同 data、同 input、明确 measurement rule。**

## 1. 概念账本

**已经知道：** **中央处理器（Central Processing Unit, CPU）**、**图形处理器（Graphics Processing Unit, GPU）**、**现场可编程门阵列（Field-Programmable Gate Array, FPGA）**、latency、throughput、bandwidth、作为工程量的 power，以及 reproducible replay。

**今天学习：** 一个**公平 benchmark contract**。

**只预告：** 真实 CPU/GPU/FPGA implementation、board power instrumentation 与完整 P-001~P-008 report。

## 2. 没有 workload 的“快”不是比较

比较 machine 前，至少冻结：

- neuron-model semantics 与 numerical precision；
- 精确 network image/version；
- initial state 与 input events；
- warm-up 与 measurement window；
- latency/throughput 到底包括什么；
- power measurement method。

如果换 machine 的同时还换 model，结果就无法解释。

## 3. benchmark contract

```mermaid
flowchart LR
  A["same model"] --> E["benchmark"]
  B["same data"] --> E
  C["same input"] --> E
  D["same measurement rules"] --> E
  E --> F["latency / throughput / energy"]
```

## 4. Run：从 synthetic measurement 计算指标

下面全部是**教学用 synthetic measurement**，不代表任何真实 CPU、GPU 或 FPGA 的性能。

In [ ]:
runs = [
    {"machine": "CPU-teaching", "events": 1_000_000, "seconds": 0.50, "watts": 40.0},
    {"machine": "GPU-teaching", "events": 1_000_000, "seconds": 0.20, "watts": 120.0},
    {"machine": "FPGA-teaching", "events": 1_000_000, "seconds": 0.25, "watts": 18.0},
]

for run in runs:
    throughput = run["events"] / run["seconds"]
    energy_j = run["watts"] * run["seconds"]
    energy_per_event = energy_j / run["events"]
    print(
        run["machine"],
        "events/s =", round(throughput),
        "J/event =", f"{energy_per_event:.8f}",
    )

## 5. Observe

不同 metric 可能给出不同方向。throughput 最高的 machine，不一定拥有最低 energy/event。

所以正式 benchmark report 应该暴露多个 metric，而不是把整个 experiment 压缩成一句模糊的“更快”。

## 6. latency 与 throughput 回答不同问题

- **latency**：一个 event/request/experiment 要多久；
- **throughput**：单位时间完成多少工作；
- **energy per event**：一个 work unit 对应多少测得 energy。

benchmark 必须先定义 work unit，这些数字才可解释。

## 7. power 特别容易比错

power 可能是 tool estimate、board-level measurement、accelerator-only measurement，也可能是 whole-system wall power。

RMD-028 要求 method 文档化。不能混用不同 measurement boundary 后，再把比值称为 architecture result。

## 8. Try It

只把 event count 加倍，而 seconds 与 watts 不变。计算出的 throughput 与 energy/event 会怎样变化？

再问：不改变 workload 或 run time，这个假设变化在物理上合理吗？

## 9. 作业

[第 23 课作业：计算 throughput 与 energy per event](../../exercises/zh/23_cpu_gpu_fpga_benchmark.ipynb)

## 10. AI Task

给 AI 两份 benchmark summary，问它们能否比较。要求它在下结论前，把所有缺失的 workload / measurement field 先列出来。

## 11. Human Check

解释为什么 “task 名字一样” 仍不足以保证 CPU/GPU/FPGA comparison 公平。哪些 field 必须冻结？为什么 latency、throughput、energy 可能指向不同方向？

## 12. Engineering Handoff

对应 `RMD-028` 与 `TRACE-P-001`。正式 benchmark claim 必须使用同 model、data、input，并保留 P-001~P-008 背后的 raw measurement evidence。

## 13. Project Trace

- Lesson：`LSN-023`
- 映射：`RMD-028`
- performance path：`TRACE-P-001`
- reproducibility path：`TRACE-F-001`
- metrics：`P-001~P-008`

## 14. Exit Ticket

你能够说明 CPU/GPU/FPGA benchmark 的最小 comparability contract，并从文档化 measurement 计算 throughput 与单位工作量 energy。